In [1]:
import pika
import pandas as pd
import json

In [2]:
import influxdb_client, os, time
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
from influxdb_client.client.exceptions import InfluxDBError

In [3]:
INFLUXDB_TOKEN="CdNR2m9lhVe4OlM2sS8BDzXVVqgEcE9RZa5vZpz5OCUnkNZnEcyS4TrFjZVtzBDYD2JjxyYY1XPP0idn9owOkg=="

In [4]:
token = INFLUXDB_TOKEN
org = "Mutual Fund Data Collection"
url = "http://127.0.0.1:8086"
bucket="mutual_fund_nav_data"
write_client = influxdb_client.InfluxDBClient(url=url, token=token, org=org)
write_api = write_client.write_api(write_options=SYNCHRONOUS)

In [5]:
# connection = pika.BlockingConnection(pika.ConnectionParameters(host='localhost'))
# channel = connection.channel()

In [9]:
connection = pika.BlockingConnection(pika.ConnectionParameters(host='localhost'))
channel = connection.channel()
class Daily_NAV_Update:
    
    def __init__(self, ROUTING_KEY, EXCHANGE_NAME, QUEUE_NAME) -> None:
        self.__ROUTING_KEY=ROUTING_KEY
        self.__EXCHANGE_NAME=EXCHANGE_NAME
        self.__QUEUE_NAME=QUEUE_NAME
        self.consumer_daily_nav_data()

    
    def write_to_influx(self, point):
        try:
            write_api.write(bucket=bucket, record=point)
        except InfluxDBError as e:
            print("Error Recieved:" , e)
    
    def parse_daily_nav_data(self, data):
            # CURRENT_DATA_POINT_VALUE=data['0']
            # print(CURRENT_DATA_POINT_VALUE)
        for key, value in data.items():
            CURRENT_DATA_POINT_VALUE=value
            # measurement=CURRENT_DATA_POINT_VALUE['amc_mutual_fund'].replace(' ', '_').lower()
            measurement=CURRENT_DATA_POINT_VALUE['amc_mutual_fund']
            point = Point(measurement_name=measurement)
            # point.tag("mutual_fund_name", CURRENT_DATA_POINT_VALUE['mutual_fund_name'].replace(' ', '_').lower())
            point.tag("mutual_fund_name", CURRENT_DATA_POINT_VALUE['mutual_fund_name'])
            point.tag("db_record_time", CURRENT_DATA_POINT_VALUE['timestamp'])
            point.time(CURRENT_DATA_POINT_VALUE['nav_upload_date_time'])
            point.field("nav", float(CURRENT_DATA_POINT_VALUE['nav']))
            print("Point Record Written:", str(point))
            self.write_to_influx(point)
            

    def daily_nav_data_consumer_callback(self, ch, method, properties, body):
        print("Recieved Message")
        # print(" [x] %r:%r" % (method.routing_key, body))
        self.parse_daily_nav_data(json.loads(body.decode('utf8')))
        
    def consumer_daily_nav_data(self):
        channel.basic_consume(queue=self.__QUEUE_NAME, on_message_callback=self.daily_nav_data_consumer_callback, auto_ack=True)

    
daily_nav_update=Daily_NAV_Update("daily_nav_data", 'mutual_fund_data_collection', 'daily_nav_data_queue')

In [11]:
channel.start_consuming()